<a href="https://colab.research.google.com/github/Rhea-Shah23/indiancine.ma-scrape-curate/blob/main/MACS207_Indiancine_ma_Scrape_and_Curate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Group 6:** Rhea Shah, Disha Dayalan, Navya Dixit

**Project Objectives:**


*   Our Resource: https://indiancine.ma/
*   Our Prompt: Scrape and Curate Only Titles, Years, and IMDdID






In [ ]:
!pip install playwright nest_asyncio

In [ ]:
!playwright install chromium

In [ ]:
# Use Playwright to scrape dynamic content from Indian Cine.ma
import asyncio
from playwright.async_api import async_playwright
import pandas as pd
import re

# Allows async Playwright code to run properly inside Google Colab/Jupyter
import nest_asyncio
nest_asyncio.apply()

# Function to scrape data using Playwright
async def scrape_with_playwright():
    movies = []

    async with async_playwright() as p:
        # Opens browser in the background
        browser = await p.chromium.launch(headless=True)  # Set headless=False to see the browser
        page = await browser.new_page()

        await page.goto("https://indiancine.ma/grid/year")

        # Wait for the page to load the movie elements
        await page.wait_for_selector('.OxElement')

        await asyncio.sleep(5)

        # 1st pass collects title, year, and Indian Cine.ma movie ID from each grid item
        movie_data = []
        movie_elements = await page.query_selector_all('.OxItem')

        for movie in movie_elements:
            try:
                # Get title (OxTarget text, minus the year span)
                title_element = await movie.query_selector('.OxTarget')
                year_element = await movie.query_selector('.OxInfo')
                img_element = await movie.query_selector('.OxIcon img')

                if not title_element or not year_element or not img_element:
                    continue

                title_raw = await title_element.inner_text()
                year = (await year_element.text_content()).strip()

                # The poster/img URL has the movie ID
                img_src = await img_element.get_attribute('src')
                id_match = re.search(r'indiancine\.ma/([^/]+)/poster', img_src)
                movie_id = id_match.group(1).upper() if id_match else None

                # Clean title by removing the year, extra spaces, nd directors name
                title = re.sub(r'\s*' + year + r'\s*', '', title_raw).strip()
                title = re.sub(r'\s+', ' ', title)
                title = re.sub(r'\s*\([^)]*\)', '', title).strip()

                movie_data.append({"title": title, "year": year, "id": movie_id})

            except Exception as e:
                print(f"Error reading a grid item: {e}")

        print(f"Collected {len(movie_data)} movies from grid")

        # 2nd pass: visit each movie's info page to collect the IMDb ID
        for i, m in enumerate(movie_data):
            try:
                if not m["id"]:
                    movies.append({"Title": m["title"], "Year": m["year"], "IMDbID": "N/A"})
                    continue

                movie_url = f"https://indiancine.ma/{m['id']}/info"
                await page.goto(movie_url)
                await asyncio.sleep(2)

                # search page text for IMDb ID
                content = await page.text_content('body')
                imdb_match = re.search(r'IMDb ID[:\s]+(\d+)', content)
                imdb_id = imdb_match.group(1) if imdb_match else "N/A"

                movies.append({
                    "Title": m["title"],
                    "Year": m["year"],
                    "IMDbID": imdb_id
                })

                print(f"[{i+1}/{len(movie_data)}] {m['title']} ({m['year']}) → {imdb_id}")

            except Exception as e:
                print(f"Error on {m['title']}: {e}")
                movies.append({
                    "Title": m["title"],
                    "Year": m["year"],
                    "IMDbID": "N/A"
                })

        await browser.close()

    # Create and display the final table
    df = pd.DataFrame(movies)
    display(df)

    return df

# Run the function and store table
df = await scrape_with_playwright()

Collected 126 movies from grid
[1/126] Cocoanut Fair (1897) → 0214594
[2/126] Our Indian Empire (1897) → 0392570
[3/126] Dancing Scenes from the Flower of Persia (1898) → 0319248
[4/126] A Panorama of Indian Scenes and Procession (1898) → 0368121
[5/126] Man and Monkey (1899) → 0155850
[6/126] The Wrestlers (1899) → 0156186
[7/126] Moving Pictures of Natural Scenes and Religious Rituals (1899) → 0273846
[8/126] Local Scenes Bombay (1899) → 0392351
[9/126] Panorama of Calcutta (1899) → 0320236
[10/126] Splendid New Views of Bombay (1900) → 0393811
[11/126] Taboot Procession at Kalbadevi (1900) → 0393845
[12/126] Atash Behram (1901) → 0273438
[13/126] Landing of Sir M.M. Bhownuggree (1901) → 0155828
[14/126] Scenes from Alibaba (1901) → 0332747
[15/126] Scenes from Bhramar (1901) → 0326113
[16/126] Scenes from Buddhadev (1901) → 0332748
[17/126] Scenes from 'Dol Leela' (1901) → 0332749
[18/126] Scenes from Hariraj (1901) → 0332750
[19/126] Scenes from Sarala (1901) → 0332752
[20/126] Sce

,Title,Year,IMDbID
0,Cocoanut Fair,1897,0214594
1,Our Indian Empire,1897,0392570
2,Dancing Scenes from the Flower of Persia,1898,0319248
3,A Panorama of Indian Scenes and Procession,1898,0368121
4,Man and Monkey,1899,0155850
...,...,...,...
121,Congress Session in Bombay,1919,0249450
122,Kabir Kamal,1919,0251165
123,Kacha Devayani,1919,0251166
124,Narasinh Avatar,1919,0251288
